In [1]:
# Instalações para Google Colab
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium pandas openpyxl > /dev/null

# Imports
import os, re, time, requests, zipfile
from datetime import datetime
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# Função para limpar strings
def limpar(texto):
    return ' '.join(texto.strip().split()) if texto else "N/A"

# Setup de pastas com data
data_hoje = datetime.now().strftime('%Y-%m-%d')
pasta_base = f"{data_hoje}_casa_rossi"
pasta_imagens = os.path.join(pasta_base, "imagens")
os.makedirs(pasta_imagens, exist_ok=True)

# Configurar WebDriver (headless)
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)

# Coletar links de todas as páginas
def coletar_links_produtos():
    url_base = 'https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado'
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url_base, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Descobrir o número total de páginas
    ultima_pagina_tag = soup.select_one("li.page.last a")
    total_paginas = int(re.search(r'pg=(\d+)', ultima_pagina_tag['href']).group(1)) if ultima_pagina_tag else 1
    print(f"📄 Total de páginas: {total_paginas}")

    todos_links = []
    for pagina in range(1, total_paginas + 1):
        print(f"🔄 Página {pagina}/{total_paginas}")
        url_pagina = f"{url_base}?pg={pagina}"
        response = requests.get(url_pagina, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        produtos = soup.find_all('li', class_='item flex')

        for produto in produtos:
            a_tag = produto.find('a', class_='space-image')
            if a_tag:
                link = a_tag['href']
                if not link.startswith("http"):
                    link = "https://www.casarossi.com.br" + link
                todos_links.append(link)

        time.sleep(1)  # evitar sobrecarga no servidor

    print(f"🔗 Total de produtos encontrados: {len(todos_links)}")
    return todos_links

# Função para salvar imagem
def salvar_imagem(url_img, nome_base):
    if url_img == "N/A":
        return
    try:
        resposta = requests.get(url_img, timeout=10)
        if resposta.status_code == 200:
            caminho = os.path.join(pasta_imagens, f"{nome_base}.jpg")
            with open(caminho, 'wb') as f:
                f.write(resposta.content)
    except Exception as e:
        print(f"❌ Erro ao baixar imagem: {e}")

# Coletar links dos produtos
links_produtos = coletar_links_produtos()

# Extrair dados
dados_produtos = []
for i, url in enumerate(links_produtos):
    print(f"[{i+1}/{len(links_produtos)}] Coletando dados de: {url}")
    try:
        driver.get(url)
        time.sleep(3)
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        nome = limpar(soup.find('h1', class_='product-name').text)
        codigo = limpar(soup.select_one('.product-sku .ref').text) if soup.select_one('.product-sku .ref') else nome[:20]
        marca = limpar(soup.find_all('div', class_='product-brand')[0].text.replace('Marca:', '')) if soup.find_all('div', class_='product-brand') else "N/A"
        modelo = limpar(soup.find_all('div', class_='product-brand')[1].text.replace('Modelo:', '')) if len(soup.find_all('div', class_='product-brand')) > 1 else "N/A"
        preco_caixa = limpar(soup.select_one('.PrecoPrincipal span').text) if soup.select_one('.PrecoPrincipal span') else "N/A"
        preco_m2 = limpar(soup.select_one('.preco-por-metro').text.replace('por m²', '').replace('R$', '').strip()) if soup.select_one('.preco-por-metro') else "N/A"
        m2_caixa = limpar(soup.select_one('.metro-na-caixa').text.replace('Caixa com', '').replace('m²', '').strip()) if soup.select_one('.metro-na-caixa') else "N/A"
        imagem_tag = soup.select_one('.zoom img')
        imagem_url = imagem_tag['src'] if imagem_tag else "N/A"
        descricao_html = soup.find('p')
        descricao = limpar(descricao_html.get_text(separator=' ')) if descricao_html else "N/A"

        salvar_imagem(imagem_url, codigo)

        dados_produtos.append([
            codigo,
            nome,
            marca,
            modelo,
            preco_m2,
            preco_caixa,
            m2_caixa,
            f"{codigo}.jpg",
            descricao,
            imagem_url,
            url,
            "Casa Rossi"
        ])
    except Exception as e:
        print(f"❌ Erro ao processar {url}: {e}")

driver.quit()

# Criar DataFrame
df = pd.DataFrame(dados_produtos, columns=[
    "Código", "Nome", "Marca", "Modelo", "Preço por m²", "Preço por Caixa",
    "m² por Caixa", "Imagem", "Descrição", "URL Imagem", "URL Produto", "Loja"
])

# Função para limpar preços
def limpar_preco(valor):
    try:
        return float(valor.replace("R$", "").replace(".", "").replace(",", ".").strip())
    except:
        return None

df["Preço por m²"] = df["Preço por m²"].apply(limpar_preco)
df["Preço por Caixa"] = df["Preço por Caixa"].apply(limpar_preco)

# Salvar CSV
csv_path = os.path.join(pasta_base, f"casa_rossi_{data_hoje}.csv")
df.to_csv(csv_path, sep=';', index=False, encoding='utf-8-sig')
print(f"📄 CSV salvo em: {csv_path}")

# Salvar Excel
xlsx_path = os.path.join(pasta_base, f"casa_rossi_{data_hoje}.xlsx")
df.to_excel(xlsx_path, index=False)
print(f"📊 Excel salvo em: {xlsx_path}")

# Compactar imagens
zip_path = os.path.join(pasta_base, f"imagens_casa_rossi_{data_hoje}.zip")
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk(pasta_imagens):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=file)

print(f"📦 Imagens compactadas em: {zip_path}")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
📄 Total de páginas: 8
🔄 Página 1/8
🔄 Página 2/8
🔄 Página 3/8
🔄 Página 4/8
🔄 Página 5/8
🔄 Página 6/8
🔄 Página 7/8
🔄 Página 8/8
🔗 Total de produtos encontrados: 110
[1/110] Coletando dados de: https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/piso-formigres-66x66cm-acetinado-retificado-coimbra-2-18m
[2/110] Coletando dados de: https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/porcelanato-retificado-delta-72x72cm-out-itaara-raven-2-61m
[3/110] Coletando dados de: https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/porcelanatoincepa-90x90cm-acetinado-duo-concret-cris-2-40m
[4/110] Coletando dados de: https://www.casarossi.com.br/pisos-e-revesti